# Trust Game: TRL + Unsloth Training

This notebook trains a policy model that maps Trust Game prompts to JSON actions and evaluates reward improvements before/after fine-tuning.

**Recommended runtime:** Google Colab (Linux) with GPU (T4 or better), Python 3.10/3.11. Local macOS + Python 3.13 may hit wheel availability and timeout issues for Torch/Unsloth.

In [1]:
# Setup (best on Colab Linux + GPU, Python 3.10/3.11)
from pathlib import Path
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Install ML deps (retry/timeout guards help flaky networks)
%pip install -q --retries 10 --timeout 120 unsloth trl transformers datasets accelerate peft bitsandbytes matplotlib pandas

# Install this project from repo root (not training/)
repo_root = Path.cwd().resolve().parent
%pip install -q -e "{repo_root}"

ERROR: file:///Users/hardik/The%20Trust%20Game/trust_game_env/training does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
import json
import random

import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset

from trust_game_env.server.trust_game_env_environment import TrustGameEnvironment
from trust_game_env.baselines.policies import policy_for_role
from trust_game_env.models import TrustGameAction

from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)

In [ ]:
def action_to_json(action: TrustGameAction) -> str:
    payload = {
        "agent_id": int(action.agent_id),
        "claim_amount": float(action.claim_amount),
        "verify_targets": list(action.verify_targets),
        "accept_proposal": bool(action.accept_proposal),
    }
    return json.dumps(payload)

def safe_parse_action(text: str, fallback_agent_id: int) -> TrustGameAction:
    try:
        left = text.find('{')
        right = text.rfind('}')
        obj = json.loads(text[left:right+1])
        return TrustGameAction(
            agent_id=int(obj.get('agent_id', fallback_agent_id)),
            claim_amount=float(obj.get('claim_amount', 40.0)),
            verify_targets=[int(x) for x in obj.get('verify_targets', [])],
            accept_proposal=bool(obj.get('accept_proposal', False)),
        )
    except Exception:
        return TrustGameAction(
            agent_id=fallback_agent_id,
            claim_amount=40.0,
            verify_targets=[],
            accept_proposal=False,
        )

def generate_dataset(n_episodes: int = 300):
    rows = []
    for seed in range(n_episodes):
        env = TrustGameEnvironment(curriculum_stage=2, seed=seed)
        obs = env.reset()
        done = False
        while not done:
            role = env.roles[obs.your_agent_id]
            action = policy_for_role(role).act(obs)
            rows.append({
                'prompt': obs.prompt,
                'response': action_to_json(action),
            })
            obs, _, done = env.step(action)
    return Dataset.from_pandas(pd.DataFrame(rows))

dataset = generate_dataset(250)
dataset = dataset.train_test_split(test_size=0.1, seed=SEED)
dataset

In [ ]:
MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

def format_example(example):
    text = (
        'You are an agent in the Trust Game environment.\n'
        'Return ONLY valid JSON with keys: agent_id, claim_amount, verify_targets, accept_proposal.\n\n'
        f"Prompt:\n{example['prompt']}\n\nAction JSON:\n{example['response']}"
    )
    return {'text': text}

train_ds = dataset['train'].map(format_example)
eval_ds = dataset['test'].map(format_example)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir='outputs/trust-game-sft',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_steps=100,
        bf16=True,
        report_to='none',
    ),
    dataset_text_field='text',
)

train_output = trainer.train()
train_output

In [ ]:
def model_action(obs, max_new_tokens=120):
    prompt = (
        'Return ONLY valid JSON with keys: agent_id, claim_amount, verify_targets, accept_proposal.\n\n'
        f"Prompt:\n{obs.prompt}\n\nAction JSON:\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return safe_parse_action(text, fallback_agent_id=obs.your_agent_id)

def scripted_action(env, obs):
    return policy_for_role(env.roles[obs.your_agent_id]).act(obs)

def run_eval(policy_kind='scripted', n_episodes=30):
    rewards = []
    for seed in range(1000, 1000 + n_episodes):
        env = TrustGameEnvironment(curriculum_stage=2, seed=seed)
        obs = env.reset()
        done = False
        while not done:
            action = scripted_action(env, obs) if policy_kind == 'scripted' else model_action(obs)
            obs, _, done = env.step(action)
        rewards.append(sum(env.episode_rewards.values()) / len(env.episode_rewards))
    return rewards

baseline_rewards = run_eval('scripted', n_episodes=20)
trained_rewards = run_eval('model', n_episodes=20)

print('Baseline mean reward:', sum(baseline_rewards)/len(baseline_rewards))
print('Trained mean reward :', sum(trained_rewards)/len(trained_rewards))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(baseline_rewards, label='Baseline', marker='o')
plt.plot(trained_rewards, label='Trained', marker='o')
plt.xlabel('Episode')
plt.ylabel('Mean episode reward')
plt.title('Trust Game: Baseline vs Trained')
plt.legend()
plt.tight_layout()

import os
os.makedirs('eval_results', exist_ok=True)
plot_path = 'eval_results/training_reward_comparison.png'
plt.savefig(plot_path, dpi=180)
print('Saved plot to', plot_path)

In [ ]:
trainer.save_model('outputs/trust-game-sft/final')
tokenizer.save_pretrained('outputs/trust-game-sft/final')
print('Saved model to outputs/trust-game-sft/final')